In [1]:
from pathlib import Path

import nibabel as nib
import numpy as np
import pandas as pd

from nibabel.processing import resample_to_output
from resources.cut_off_edges import cut_off_edges
from resources.standarize_image import standardize_image

In [2]:
project_dir = Path.cwd().parents[1]

# Jeżeli notebook znajduje się np. w katalogu src:
if not (project_dir / "data").exists():
    project_dir = project_dir.parent

niftis_dir = project_dir / "data" / "niftis"
labels_path = niftis_dir / "train_labels_JNDlMjr.csv"

outputs_dir = project_dir / "src" / "outputs"
outputs_dir.mkdir(parents=True, exist_ok=True)

labels_df = pd.read_csv(labels_path)

target_column = "is_pathologic"

labels = labels_df.set_index("uid")[target_column]

nifti_paths = sorted(niftis_dir.glob("*.nii.gz"))

print("Katalog projektu:", project_dir)
print("Liczba obrazów:", len(nifti_paths))
print("Liczba etykiet:", len(labels))

Katalog projektu: C:\Users\abram\PycharmProjects\DrivenData_DAT_Parkinsons
Liczba obrazów: 1362
Liczba etykiet: 1362


In [3]:
volumes = []
y_labels = []
uids = []

for index, nifti_path in enumerate(nifti_paths, start=1):
    uid = nifti_path.name.removesuffix(".nii.gz")

    if uid not in labels.index:
        print(f"Brak etykiety dla {uid}")
        continue

    print(f"{index}/{len(nifti_paths)}: {uid}")

    image = nib.load(nifti_path)

    processed_image = cut_off_edges(image)

    processed_image = cut_off_edges(
        processed_image,
        x_left=35,
        x_right=35,
        y_front=40,
        y_back=40,
        z_down=30,
        z_up=30,
    )

    processed_image = standardize_image(
        processed_image,
        percent_top=0.02,
    )

    processed_image = resample_to_output(
        processed_image,
        voxel_sizes=(2.46, 2.46, 2.46),
        order=1,
    )

    processed_image = cut_off_edges(
        processed_image,
        x_left=30,
        x_right=30,
        y_front=30,
        y_back=30,
        z_down=20,
        z_up=25,
    )
    processed_image = cut_off_edges(
        processed_image,
        x_left=25,
        x_right=25,
        y_front=30,
        y_back=10,
        z_down=10,
        z_up=25,
    )
    volume = processed_image.get_fdata(dtype=np.float32)
    volumes.append(volume)
    y_labels.append(int(labels.loc[uid]))
    uids.append(uid)

1/1362: 01nouhtc
2/1362: 0224wk0y
3/1362: 049enulq
4/1362: 04x0qzfh
5/1362: 05xt54fn
6/1362: 06akt7zq
7/1362: 06nkwfg3
8/1362: 07gbasf5
9/1362: 07s387j5
10/1362: 085rntdr
11/1362: 08liuqdn
12/1362: 08uhbxlb
13/1362: 09lzrk79
14/1362: 09vs9kkh
15/1362: 0aeoorbt
16/1362: 0bq1s9ry
17/1362: 0dk0get8
18/1362: 0dl5e11e
19/1362: 0ezzf8s2
20/1362: 0fgzvrct
21/1362: 0i4x6g1e
22/1362: 0i6g61pc
23/1362: 0imjyjcm
24/1362: 0iv1ggxr
25/1362: 0j1oulwo
26/1362: 0jdcli69
27/1362: 0jf6dnob
28/1362: 0jjsp49c
29/1362: 0kb68rer
30/1362: 0l1offkh
31/1362: 0lzjjege
32/1362: 0odwmqx5
33/1362: 0pdw5py6
34/1362: 0qpop1o8
35/1362: 0s0r2sds
36/1362: 0sqcnf9w
37/1362: 0uc87nxp
38/1362: 0vgnxb5j
39/1362: 0vtkrrvw
40/1362: 0w10lbt3
41/1362: 0yknfu84
42/1362: 118qqvae
43/1362: 11bd2sce
44/1362: 11zsb8lv
45/1362: 12pgyme6
46/1362: 146vey0a
47/1362: 14nijcp5
48/1362: 153sflse
49/1362: 15ahdbvz
50/1362: 15ikpqah
51/1362: 15z8vmuh
52/1362: 167d6vnx
53/1362: 199dxj37
54/1362: 1b0z3nx3
55/1362: 1bj9rm7p
56/1362: 1c2zl46q
5

In [4]:
from collections import Counter

shape_counts = Counter(volume.shape for volume in volumes)
print(shape_counts)

Counter({(50, 40, 35): 1362})


In [5]:
X = np.stack(volumes).astype(np.float32)
y = np.asarray(y_labels, dtype=np.int64)
uids = np.asarray(uids)

print("X:", X.shape)
print("y:", y.shape)

X: (1362, 50, 40, 35)
y: (1362,)


In [6]:
X = X[:, np.newaxis, ...]

print(X.shape)

(1362, 1, 50, 40, 35)


In [7]:
np.savez_compressed(
    project_dir / "data" / "processed_volumes.npz",
    uid=uids,
    X=X.astype(np.float16),
    y=y,
)

In [8]:
data = np.load(
    project_dir / "data" / "processed_volumes.npz",
    allow_pickle=False,
)

X = data["X"].astype(np.float32)
y = data["y"].astype(np.int64)
uids = data["uid"]

In [9]:
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
indices = np.arange(len(y))

train_indices, test_indices = train_test_split(
    indices,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

X_train = X[train_indices]
X_test = X[test_indices]

y_train = y[train_indices]
y_test = y[test_indices]

uids_train = uids[train_indices]
uids_test = uids[test_indices]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (1089, 1, 50, 40, 35)
X_test: (273, 1, 50, 40, 35)


In [10]:
class VolumeDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, index):
        volume = torch.from_numpy(self.X[index]).float()
        label = torch.tensor(self.y[index], dtype=torch.float32)

        return volume, label

In [11]:
train_dataset = VolumeDataset(X_train, y_train)
test_dataset = VolumeDataset(X_test, y_test)

In [12]:
train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0,
)

validation_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0,
)

In [25]:
from torch import nn
class Simple3DCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv3d(
                in_channels=1,
                out_channels=8,
                kernel_size=3,
                padding=1,
            ),
            nn.ReLU(),
            nn.MaxPool3d(2),

            nn.Conv3d(
                in_channels=8,
                out_channels=16,
                kernel_size=3,
                padding=1,
            ),
            nn.ReLU(),
            nn.MaxPool3d(2),

            nn.Conv3d(
                in_channels=16,
                out_channels=32,
                kernel_size=3,
                padding=1,
            ),
            nn.ReLU(),

            nn.AdaptiveAvgPool3d(1),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.45),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)

        return x.squeeze(1)

In [24]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = Simple3DCNN().to(device)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=0.0001,
)

print("Urządzenie:", device)

Urządzenie: cpu


In [35]:
import torch
import torch.nn as nn


def reset_random_weights_to_zero(model, percent=0.01):
    reset_count = 0

    with torch.no_grad():
        for layer in model.modules():
            if isinstance(layer, (nn.Conv3d, nn.Linear)):
                weights = layer.weight.view(-1)

                number_to_reset = max(
                    1,
                    int(weights.numel() * percent),
                )

                selected_indices = torch.randperm(
                    weights.numel(),
                    device=weights.device,
                )[:number_to_reset]

                weights[selected_indices] = 0.0
                reset_count += number_to_reset

    return reset_count

In [44]:
max_repeat = 100
target_validation_loss = 0.22
repeat_number = 0
best_validation_loss = 0.5
while best_validation_loss > target_validation_loss and repeat_number < max_repeat:
    repeat_number += 1
    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

    # Architektura musi być taka sama jak podczas wcześniejszego treningu
    model = Simple3DCNN().to(device)

    model_path = project_dir / "models" / "simple_3d_cnn_2026-09-03_11-31-26.pth"

    model_state = torch.load(
        model_path,
        map_location=device,
        weights_only=True,
    )

    model.load_state_dict(model_state)
    reset_count = reset_random_weights_to_zero(
        model,
        percent=0.02,
    )
    criterion = nn.BCEWithLogitsLoss()

    # Nowy optimizer z mniejszym learning rate
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=0.0001,
        weight_decay=0.0001,
    )

    model.train()

    print("Wczytano:", model_path)
    print("Urządzenie:", device)

    import copy

    best_validation_loss = float("inf")
    best_model_state = None

    epochs = 30

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0

        for volumes_batch, labels_batch in train_loader:
            volumes_batch = volumes_batch.to(device)
            labels_batch = labels_batch.to(device)

            optimizer.zero_grad()

            logits = model(volumes_batch)
            loss = criterion(logits, labels_batch)

            loss.backward()
            optimizer.step()

            train_loss += loss.item() * len(labels_batch)

        train_loss /= len(train_dataset)

        model.eval()
        validation_loss = 0.0

        with torch.no_grad():
            for volumes_batch, labels_batch in validation_loader:
                volumes_batch = volumes_batch.to(device)
                labels_batch = labels_batch.to(device)

                logits = model(volumes_batch)
                loss = criterion(logits, labels_batch)

                validation_loss += loss.item() * len(labels_batch)

        validation_loss /= len(test_dataset)

        if validation_loss < best_validation_loss:
            best_validation_loss = validation_loss
            best_model_state = copy.deepcopy(model.state_dict())

        print(
            f"Epoka {epoch + 1}/{epochs} | "
            f"train: {train_loss:.4f} | "
            f"validation: {validation_loss:.4f}"
        )

    model.load_state_dict(best_model_state)

    print("Najlepszy validation loss:", best_validation_loss)

Wczytano: C:\Users\abram\PycharmProjects\DrivenData_DAT_Parkinsons\models\simple_3d_cnn_2026-09-03_11-31-26.pth
Urządzenie: cpu
Epoka 1/30 | train: 0.2601 | validation: 0.2570
Epoka 2/30 | train: 0.2380 | validation: 0.2560
Epoka 3/30 | train: 0.2199 | validation: 0.2496
Epoka 4/30 | train: 0.2268 | validation: 0.2390
Epoka 5/30 | train: 0.2246 | validation: 0.2399
Epoka 6/30 | train: 0.2250 | validation: 0.2447
Epoka 7/30 | train: 0.2099 | validation: 0.2498
Epoka 8/30 | train: 0.2223 | validation: 0.2421
Epoka 9/30 | train: 0.2339 | validation: 0.2512
Epoka 10/30 | train: 0.2068 | validation: 0.2544
Epoka 11/30 | train: 0.2161 | validation: 0.2343
Epoka 12/30 | train: 0.2222 | validation: 0.2474
Epoka 13/30 | train: 0.2248 | validation: 0.2616
Epoka 14/30 | train: 0.2112 | validation: 0.2426
Epoka 15/30 | train: 0.2113 | validation: 0.2390
Epoka 16/30 | train: 0.2142 | validation: 0.2404
Epoka 17/30 | train: 0.2136 | validation: 0.2443
Epoka 18/30 | train: 0.2122 | validation: 0.2435

KeyboardInterrupt: 

In [43]:
import copy

best_validation_loss = float("inf")
best_model_state = None

epochs = 150

for epoch in range(epochs):
    model.train()
    train_loss = 0.0

    for volumes_batch, labels_batch in train_loader:
        volumes_batch = volumes_batch.to(device)
        labels_batch = labels_batch.to(device)

        optimizer.zero_grad()

        logits = model(volumes_batch)
        loss = criterion(logits, labels_batch)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * len(labels_batch)

    train_loss /= len(train_dataset)

    model.eval()
    validation_loss = 0.0

    with torch.no_grad():
        for volumes_batch, labels_batch in validation_loader:
            volumes_batch = volumes_batch.to(device)
            labels_batch = labels_batch.to(device)

            logits = model(volumes_batch)
            loss = criterion(logits, labels_batch)

            validation_loss += loss.item() * len(labels_batch)

    validation_loss /= len(test_dataset)

    if validation_loss < best_validation_loss:
        best_validation_loss = validation_loss
        best_model_state = copy.deepcopy(model.state_dict())

    print(
        f"Epoka {epoch + 1}/{epochs} | "
        f"train: {train_loss:.4f} | "
        f"validation: {validation_loss:.4f}"
    )

model.load_state_dict(best_model_state)

print("Najlepszy validation loss:", best_validation_loss)

Epoka 1/150 | train: 0.4800 | validation: 0.2775
Epoka 2/150 | train: 0.2772 | validation: 0.2722
Epoka 3/150 | train: 0.2581 | validation: 0.2568
Epoka 4/150 | train: 0.2589 | validation: 0.2600
Epoka 5/150 | train: 0.2528 | validation: 0.2571
Epoka 6/150 | train: 0.2366 | validation: 0.2546
Epoka 7/150 | train: 0.2474 | validation: 0.2708
Epoka 8/150 | train: 0.2309 | validation: 0.2674
Epoka 9/150 | train: 0.2250 | validation: 0.2517
Epoka 10/150 | train: 0.2211 | validation: 0.2624
Epoka 11/150 | train: 0.2294 | validation: 0.2455
Epoka 12/150 | train: 0.2292 | validation: 0.2654
Epoka 13/150 | train: 0.2222 | validation: 0.2432
Epoka 14/150 | train: 0.2262 | validation: 0.2454
Epoka 15/150 | train: 0.2220 | validation: 0.2489
Epoka 16/150 | train: 0.2044 | validation: 0.2439
Epoka 17/150 | train: 0.2221 | validation: 0.2607
Epoka 18/150 | train: 0.2397 | validation: 0.2512
Epoka 19/150 | train: 0.2138 | validation: 0.2427
Epoka 20/150 | train: 0.2291 | validation: 0.2518
Epoka 21/

KeyboardInterrupt: 

In [31]:
from pathlib import Path
from datetime import datetime
import torch


models_dir = project_dir / "models"
models_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

model_path = models_dir / f"simple_relu_first_3d_cnn_{timestamp}_{best_validation_loss}.pth"

torch.save(
    model.state_dict(),
    model_path,
)

print(f"Zapisano model: {model_path} {best_validation_loss}")

Zapisano model: C:\Users\abram\PycharmProjects\DrivenData_DAT_Parkinsons\models\simple_relu_first_3d_cnn_2026-09-07_16-41-52_0.2262919164184249.pth 0.2262919164184249
